# 🚀 Fine-Tuning Qwen2.5-1.5B-Instruct for FRIDAY AI on Kaggle

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** on the full 15,000+ sample **ThisisDarkstar/friday-desktop-intent** dataset using a Kaggle free-tier GPU (Tesla T4 / P100 16GB).

### Pipeline Overview:
1. **Environment Setup**: Install PyTorch, Hugging Face `transformers`, `peft`, `datasets`, and `llama.cpp` for GGUF conversion.
2. **Dataset**: Directly stream and combine `synthetic_full` (15,000) and `calibration` (677) from Hugging Face.
3. **LoRA Fine-Tuning**: Rank 32, Alpha 64 across all linear projection layers.
4. **Weight Merging**: Fuse LoRA weights into standalone FP16 weights.
5. **GGUF Export**: Export directly to **Q8_0** (1.6 GB) and **Q4_K_M** (980 MB) GGUF files.
6. **Ollama Deployment**: Ready to download or push directly to Hugging Face.

## 1. Install Dependencies & Clone llama.cpp

In [ ]:
!pip install -q transformers peft datasets accelerate trl huggingface_hub gguf
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!pip install -q -r llama.cpp/requirements.txt

## 2. Check GPU & System Hardware

In [ ]:
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ WARNING: GPU accelerator is not enabled. In Kaggle, go to Settings -> Accelerator -> GPU T4 x2 or P100!")

## 3. Load Datasets Directly from Hugging Face Hub

In [ ]:
from datasets import load_dataset, concatenate_datasets

HF_DATASET_ID = "ThisisDarkstar/friday-desktop-intent"
print(f"[*] Loading dataset splits from Hugging Face: {HF_DATASET_ID}...")

# Load full synthetic dataset + sharp calibration boundary dataset
synthetic_ds = load_dataset(HF_DATASET_ID, "synthetic_full")
calibration_ds = load_dataset(HF_DATASET_ID, "calibration")

# Combine into full training and validation sets
train_ds = concatenate_datasets([synthetic_ds["train"], calibration_ds["train"]]).shuffle(seed=42)
val_ds = concatenate_datasets([synthetic_ds["validation"], calibration_ds["validation"]]).shuffle(seed=42)

print(f"[+] Master Training Samples: {len(train_ds):,}")
print(f"[+] Master Validation Samples: {len(val_ds):,}")

## 4. Initialize Qwen2.5-1.5B-Instruct & LoRA Adapter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./results_1.5b"
MERGED_DIR = "./friday-1.5b-merged"

print(f"[*] Loading Tokenizer for {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[*] Loading Base Model: {BASE_MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False

# High-capacity LoRA configuration (Rank 32, Alpha 64)
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Tokenize ChatML Messages with Prompt Loss Masking

In [ ]:
def format_chatml(batch, max_length=256):
    input_ids_list = []
    labels_list = []

    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        prompt_msgs = messages[:-1]
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)

        full_tokens = tokenizer(text, truncation=True, max_length=max_length)
        prompt_tokens = tokenizer(prompt_text, truncation=True, max_length=max_length)

        p_len = len(prompt_tokens["input_ids"])
        labels = [-100] * p_len + full_tokens["input_ids"][p_len:]

        if len(labels) < len(full_tokens["input_ids"]):
            labels.extend([-100] * (len(full_tokens["input_ids"]) - len(labels)))
        elif len(labels) > len(full_tokens["input_ids"]):
            labels = labels[: len(full_tokens["input_ids"])]

        input_ids_list.append(full_tokens["input_ids"])
        labels_list.append(labels)

    return {"input_ids": input_ids_list, "labels": labels_list}

print("[*] Tokenizing master training data...")
tokenized_train = train_ds.map(format_chatml, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(format_chatml, batched=True, remove_columns=val_ds.column_names)
print(f"[+] Tokenized {len(tokenized_train):,} training examples successfully!")

## 6. Execute Fine-Tuning

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,        # T4 16GB handles batch size 8 smoothly
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # Effective total batch size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=2,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    optim="adamw_torch",
    report_to="none",
)

collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=collator,
)

print(f"[*] Starting Training on {len(tokenized_train):,} samples...")
trainer.train()

## 7. Merge LoRA Weights into Standalone Model

In [ ]:
print(f"[*] Merging LoRA adapter into base model and saving to {MERGED_DIR}...")
model.config.use_cache = True
merged_model = model.merge_and_unload()
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"[+] Standalone merged model successfully saved to {MERGED_DIR}!")

## 8. Convert Merged Model to GGUF (Q8_0 & Q4_K_M)

In [ ]:
print("[*] Exporting to Q8_0 GGUF (~1.6 GB - maximum precision)...")
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype q8_0 --outfile ./friday-1.5b-q8.gguf

print("[*] Exporting to Q4_K_M GGUF (~980 MB - ultra fast inference)...")
!python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype q4_k_m --outfile ./friday-1.5b-q4km.gguf

print("\n[+] Generated GGUF Files in /kaggle/working/:")
!ls -lh *.gguf

## 9. (Optional) Upload Directly to Hugging Face Model Hub

Uncomment the cell below to upload the generated GGUFs directly to your Hugging Face account so you can pull them anywhere with `ollama run` or `huggingface-cli`.

In [ ]:
# from huggingface_hub import HfApi, login
#
# # 1. Authenticate with your write token from https://huggingface.co/settings/tokens
# login(token="hf_YOUR_TOKEN_HERE")
#
# # 2. Upload GGUFs to your Hugging Face account
# api = HfApi()
# repo_id = "ThisisDarkstar/friday-1.5b-desktop-intent-gguf"
# api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
#
# print(f"[*] Uploading friday-1.5b-q8.gguf to {repo_id}...")
# api.upload_file(
#     path_or_fileobj="friday-1.5b-q8.gguf",
#     path_in_repo="friday-1.5b-q8.gguf",
#     repo_id=repo_id,
# )
# print(f"[*] Uploading friday-1.5b-q4km.gguf to {repo_id}...")
# api.upload_file(
#     path_or_fileobj="friday-1.5b-q4km.gguf",
#     path_in_repo="friday-1.5b-q4km.gguf",
#     repo_id=repo_id,
# )
# print(f"🎉 Uploaded successfully! URL: https://huggingface.co/{repo_id}")